**Week 12**<br />Friday, Nov. 20

# Classification in NLTK

LX 394/694<br />
Coding for Linguists

Sophie Hao


## Announcements

- HW 3 is due on Monday!
- How is it going?
- Sophie forgot to record the last class (sorry!)

## Recap: Text Classification

- Spam filtering _(spam or ham?)_
- Sentiment analysis _(positive or negative?)_
- Hate speech detection _(harmful or safe?)_
- Language identification _(English, Swahili, or Quechua?)_
- Topic modeling _(business, politics, sports, or arts?)_
- Natural language inference _(entailment, contradiction, or neutral?)_
- Technically, language modeling _(a **ton** of possible labels)_

## Recap: Bayesian Classification

Let $C$ be a set of **labels** ("classes"). For a text $x$, a Bayesian classifier generates a **predicted label** $\hat{y} \in C$ as follows:

$$\hat{y} = \underset{c \in C}{\text{argmax}} \log_2(\mathbb{P}[x \mathrel{|} c]) + \log_2(\mathbb{P}[c])$$

where
- $\log_2(\mathbb{P}[x \mathrel{|} c])$ is estimated using a language model 
- $\log_2(\mathbb{P}[c])$ is the proportion of the training corpus with label $c$

## Recap: Bayes's Theorem

The Bayesian classification score comes from **Bayes's theorem**:

$$\mathbb{P}[c \mathrel{|} x] = \frac{\mathbb{P}[x \mathrel{|} c] \mathbb{P}[c]}{\mathbb{P}[x]}$$

- $\mathbb{P}[c]$ is called the **prior probability** of $c$
- $\mathbb{P}[x \mathrel{|} c]$ is called the **likelihood** of $x$ given $c$
- $\mathbb{P}[c \mathrel{|} x]$ is called the **posterior probability** of $c$ given $x$

## Naïve Bayes Features

Naïve Bayes features are not limited to bag-of-words features.

- POS tags
- word stems (i.e., excluding affixes)
- values from sentiment lexicon (for sentiment analysis)
- **Exercise:** What are some others? (for spam filtering)
- **Exercise:** Can we use these features if likelihood is calculated using a bigram model?

## Classification in NLTK

Let's train a naïve Bayes classifier on NLTK's **movie reviews corpus**.

In [1]:
import nltk
from nltk.corpus import movie_reviews

nltk.download("movie_reviews")

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/sophie/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [2]:
# Inspect the document labels
movie_reviews.categories()

['neg', 'pos']

In [3]:
# Each file is associated with one of the labels
movie_reviews.fileids("neg")[:5]

['neg/cv000_29416.txt',
 'neg/cv001_19502.txt',
 'neg/cv002_17424.txt',
 'neg/cv003_12683.txt',
 'neg/cv004_12641.txt']

In [4]:
movie_reviews.fileids("pos")[:5]

['pos/cv000_29590.txt',
 'pos/cv001_18431.txt',
 'pos/cv002_15918.txt',
 'pos/cv003_11664.txt',
 'pos/cv004_11636.txt']

## Exercise

What task do you think this is? (What do "neg" and "pos" mean?)

In [5]:
# Inspect a "neg" review
print(movie_reviews.raw("neg/cv506_17521.txt"))

this film is extraordinarily horrendous and i'm not going to waste any more words on it . 



In [6]:
# Inspect a "pos" review
print(movie_reviews.raw("pos/cv754_7216.txt"))

kolya is one of the richest films i've seen in some time . 
zdenek sverak plays a confirmed old bachelor ( who's likely to remain so ) , who finds his life as a czech cellist increasingly impacted by the five-year old boy that he's taking care of . 
though it ends rather abruptly-- and i'm whining , 'cause i wanted to spend more time with these characters-- the acting , writing , and production values are as high as , if not higher than , comparable american dramas . 
this father-and-son delight-- sverak also wrote the script , while his son , jan , directed-- won a golden globe for best foreign language film and , a couple days after i saw it , walked away an oscar . 
in czech and russian , with english subtitles . 



## Feature Extraction

In [10]:
# Get the 2000 most common words in the corpus
vocab, _ = zip(*nltk.FreqDist(movie_reviews.words()).most_common(2000))

def get_features(x: list[str]) -> dict[str, bool]:
    """Converts a movie review into feature bundles"""
    x_words = set(x)
    x_features = dict()
    for word in vocab:
        x_features[f"contains({word})"] = word in x_words
    return x_features

In [11]:
# Example of a feature bundle
get_features(movie_reviews.words("pos/cv754_7216.txt"))

{'contains(,)': True,
 'contains(the)': True,
 'contains(.)': True,
 'contains(a)': True,
 'contains(and)': True,
 'contains(of)': True,
 'contains(to)': True,
 "contains(')": True,
 'contains(is)': True,
 'contains(in)': True,
 'contains(s)': True,
 'contains(")': False,
 'contains(it)': True,
 'contains(that)': True,
 'contains(-)': True,
 'contains())': True,
 'contains(()': True,
 'contains(as)': True,
 'contains(with)': True,
 'contains(for)': True,
 'contains(his)': True,
 'contains(this)': True,
 'contains(film)': True,
 'contains(i)': True,
 'contains(he)': True,
 'contains(but)': False,
 'contains(on)': False,
 'contains(are)': True,
 'contains(t)': False,
 'contains(by)': True,
 'contains(be)': False,
 'contains(one)': True,
 'contains(movie)': False,
 'contains(an)': True,
 'contains(who)': True,
 'contains(not)': True,
 'contains(you)': False,
 'contains(from)': False,
 'contains(at)': False,
 'contains(was)': False,
 'contains(have)': False,
 'contains(they)': False,
 'con

## Construct Training and Test Sets

In [13]:
import random

# Assign gold labels to each movie review
pos_reviews = [(list(movie_reviews.words(f)), "pos") for f in movie_reviews.fileids("pos")]
neg_reviews = [(list(movie_reviews.words(f)), "neg") for f in movie_reviews.fileids("neg")]

# Put the reviews together and shuffle
reviews = pos_reviews + neg_reviews
random.shuffle(reviews)

# Convert each review into a feature bundle and split into train and test
train_set = [(get_features(x), y) for x, y in reviews[100:]]
test_set = [(get_features(x), y) for x, y in reviews[:100]]

## Train and Test a Classifier

In [14]:
# Train a classifier
classifier = nltk.NaiveBayesClassifier.train(train_set)

In [17]:
# Measure its accuracy
nltk.classify.accuracy(classifier, test_set)

0.88

## Most Informative Features

The **informativity** of a feature is measured by:
$$\max\left( \frac{\mathbb{P}[\text{contains}(w) = \text{True} \mathrel{|} \text{pos}]}{\mathbb{P}[\text{contains}(w) = \text{True} \mathrel{|} \text{neg}]}, \frac{\mathbb{P}[\text{contains}(w) = True \mathrel{|} \text{neg}]}{\mathbb{P}[\text{contains}(w) = True \mathrel{|} \text{pos}]} \right)$$

In [18]:
# Qualitative evaluation
classifier.show_most_informative_features(5)

Most Informative Features
   contains(outstanding) = True              pos : neg    =     11.3 : 1.0
   contains(wonderfully) = True              pos : neg    =      9.9 : 1.0
         contains(mulan) = True              pos : neg    =      8.3 : 1.0
         contains(damon) = True              pos : neg    =      5.9 : 1.0
          contains(lame) = True              neg : pos    =      5.6 : 1.0


**Exercise:** Does this remind you of anything?

## Next Time

Evaluating a Classifier